In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 核心优化1：只用1个Linear生成QKV，然后chunk拆开
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x, mask=None):
        # PyTorch 广播从右往左对齐，长度为 1 或缺失的维度自动扩展
        B, L, _ = x.shape  # [Batch, Seq_len, Dim]
        
        # ----- 1. 投影 + 拆分 (比三个Linear更快，内存连续) -----
        qkv = self.qkv(x)  # [B, L, 3*d_model]
        q, k, v = torch.chunk(qkv, 3, dim=-1)  # 每个 [B, L, d_model]
        
        # ----- 2. 变形为多头 (利用reshape直接展开) -----
        # 目标形状: [B, heads, L, d_k]
        q = q.reshape(B, L, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        k = k.reshape(B, L, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        v = v.reshape(B, L, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        
        # ----- 3. Einsum 注意力 (维度一目了然) -----
        # b: batch, h: heads, i/j: seq_len, d: feature_dim
        attn = torch.einsum('b h i d, b h j d -> b h i j', q, k) / (self.d_k ** 0.5)
        
        # Mask 处理 (广播机制自动适配)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
            
        attn = F.softmax(attn, dim=-1)
        attn = F.dropout(attn, p=self.dropout, training=self.training)
        
        # 加权求和: 'b h i j' 对 'b h j d' 做矩阵乘法 -> 'b h i d'
        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        
        # ----- 4. 恢复维度 (permute + reshape) -----
        out = out.permute(0, 2, 1, 3).reshape(B, L, self.d_model)
        
        return self.proj(out)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_head, dropout=0.0):
        super().__init__()
        assert d_model % num_head == 0, "d_model must be divisible by num_head"
        
        self.d_model = d_model
        self.num_head = num_head
        self.d_k = d_model // num_head
        
        # 最后一维 D 映射为 3D，供后续拆分 Query、Key、Value。
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # x: [B, L, D]，分别为批大小、序列长度和 d_model。
        B, L, D = x.shape
        
        # 1. Linear: [B, L, D] -> [B, L, 3D]；reshape 后显式分出 Q/K/V 与头维。
        # qkv: [B, L, 3, H, d_k] -> permute -> [3, B, H, L, d_k]
        qkv = self.qkv(x).reshape(B, L, 3, self.num_head, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # 每个都是 [B, H, L, d_k]
        
        # 2. 计算缩放点积注意力 (推荐用 matmul 替代 einsum)
        # q: [B, H, L, d_k] @ k^T: [B, H, d_k, L] -> attn: [B, H, L, L]
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # attn_score=torch.einsum("b h i d,b h j d->b h i j",q,k)/(self.d_k**0.5)
        
        # 3. 处理 Mask (面试加分项：兼容多种 mask 形状)
        if mask is not None:
            # mask 为 [B, L] 时扩展为 [B, 1, 1, L]，广播到每个头和每个 query 位置。
            # mask 为 [L, L] 时同样扩展，可广播到批次与头维。
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)  # [B, 1, 1, L] 或 [1, 1, L, L]
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)  # [B, 1, L, L]
            
            attn_score = attn_score.masked_fill(mask == 0, -float('inf'))
        
        # 4. 在最后一个 L（key）维归一化，注意力权重形状仍为 [B, H, L, L]。
        attn = F.softmax(attn_score, dim=-1)
        attn = self.dropout(attn)  # 形状保持 [B, H, L, L]
        
        # 5. 加权求和并合并多头
        # attn: [B, H, L, L] @ v: [B, H, L, d_k] -> out: [B, H, L, d_k]
        out = torch.matmul(attn, v)
        # out=torch.einsum("b h i j, b h j d->b h i d",attn,v)
        # 合并多头: [B, H, L, d_k] -> [B, L, H, d_k] -> [B, L, D]
        out = out.transpose(1, 2).reshape(B, L, D)
        
        # 6. 输出投影只作用于最后一维 D，形状保持 [B, L, D]。
        return self.out_proj(out)

In [ ]:
#@save
def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状"""
    # 输入X的形状:(batch_size，查询或者“键－值”对的个数，num_hiddens)
    # 输出X的形状:(batch_size，查询或者“键－值”对的个数，num_heads，num_hiddens/num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)

    # 最终输出的形状:(batch_size*num_heads,查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])


#@save
def transpose_output(X, num_heads):
    """逆转transpose_qkv函数的操作"""
    # 输入X的形状:(batch_size*num_heads，查询的个数，num_hiddens/num_heads)
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数,num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)
    # (batch_size，查询或者“键－值”对的个数，num_heads，num_hiddens/num_heads)
    return X.reshape(X.shape[0], X.shape[1], -1)
    # (batch_size，查询或者“键－值”对的个数，num_hiddens)

import math
import torch.nn.functional as F

#@save
class DotProductAttention(nn.Module):
    """缩放点积注意力（masked_fill 版本）
    
    多头已被 transpose_qkv 折叠进 batch 维，所以这里处理的是 3D 张量：
    queries: (BH, L_q, d)  keys/values: (BH, L_k, d)
    valid_lens: (BH,) 或 (BH, L_q) —— 每个 query 的有效 key 个数
    """
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        # transpose_b 效果：交换 keys 后两维 -> bmm 得到 scores (BH, L_q, L_k)
        # scores 的语义：第 i 行第 j 列 = query 位置 i 对 key 位置 j 的相关度
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)

        if valid_lens is not None:
            BH, L_q, L_k = scores.shape
            if valid_lens.dim() == 1:
                # (BH,) -> (BH*L_q,)：同一行的所有 query 共享有效长度
                valid_lens = valid_lens.repeat_interleave(L_q)
            # 核心广播技巧：列下标 arange(L_k) 与 valid_lens 列向量比较
            # (1, L_k) < (BH*L_q, 1) -> (BH*L_q, L_k)，前 valid_len 列为 True
            j = torch.arange(L_k, device=scores.device).reshape(1, -1)
            mask = (j < valid_lens.reshape(-1, 1)).reshape(scores.shape)
            # mask: (BH, L_q, L_k)，True=有效/可见；False 位置填 -inf
            scores = scores.masked_fill(~mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)   # exp(-inf)=0 -> 无效列权重归零
        self.attention_weights = attn      # 留作可视化用
        return torch.bmm(self.dropout(attn), values)


#@save
class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        # 补全：每个头共享同一个缩放点积注意力模块
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # queries，keys，values的形状:
        # (batch_size，查询或者“键－值”对的个数，num_dim)
        # valid_lens　的形状:
        # (batch_size，)或(batch_size，查询的个数)
        # self.W_q(queries)的形状:
        # (batch_size，个数，num_hiddens)
        # 经过变换后，输出的queries，keys，values　的形状:
        # (batch_size*num_heads，查询或者“键－值”对的个数，num_hiddens/num_heads)
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)

        if valid_lens is not None:
            # (batch_size，查询的个数)
            # 在轴0，将第一项（标量或者矢量）复制num_heads次，
            # 然后如此复制第二项，然后诸如此类。
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)
            # (batch_size*num_heads，查询的个数)
        
        output = self.attention(queries, keys, values, valid_lens)
        # output的形状:(batch_size*num_heads，查询的个数，num_hiddens/num_heads)
        # attention_weight形状：(batch_size*num_heads，查询的个数，“键－值”对的个数）

        # output_concat的形状:(batch_size，查询的个数，num_hiddens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)
        #返回 (batch_size，查询的个数，num_hiddens)

In [ ]:
# ===== 测试：形状 + mask 数值验证 =====
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens,
                               num_hiddens, num_heads, 0.5)
attention.eval()

batch_size, num_queries = 2, 4
num_kvpairs, valid_lens = 6, torch.tensor([3, 2])   # 样本0前3个key有效，样本1前2个
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
out = attention(X, Y, Y, valid_lens)
print('输出形状:', out.shape)          # (2, 4, 100)

# 注意力权重: (batch*num_heads, num_queries, num_kvpairs) = (10, 4, 6)
w = attention.attention.attention_weights
print('权重形状:', tuple(w.shape))

# 验证1: 每行权重和为1（softmax 归一化不被 mask 破坏）
print('每行和为1:', torch.allclose(w.sum(-1), torch.ones_like(w.sum(-1)), atol=1e-6))

# 验证2: 输入全1 -> 有效列均分 1/valid_len，无效列严格为0
print('\n样本0某头第1行 (期望前3列=1/3, 后3列=0):')
print(w[0, 0])
print('\n样本1某头第1行 (期望前2列=1/2, 后4列=0):')
print(w[batch_size*num_heads, 0] if False else w[5, 0])

## Mask 详解

### 1. `masked_fill` 在做什么

```python
attn = attn.masked_fill(mask == 0, float('-inf'))
```

逐元素替换：`condition` 为 `True` 的位置填 `value`，其余不变。
约定：**mask 中 1 = 可见/有效，0 = 屏蔽**（与 HuggingFace `attention_mask` 一致）。
被填成 `-inf` 的位置经过 `softmax` 后 `exp(-inf)=0`，注意力权重归零，即"看不到"。

### 2. Mask 的各种形状及其意义

`attn` 的形状是 `[B, H, L, L]`，mask **不需要**也是 `[B,H,L,L]`，只要满足广播规则即可：
PyTorch 广播从右往左对齐，长度为 1 或缺失的维度自动扩展成目标大小。

mask 四个维度的含义：`[B, H, i, j]` = "**哪个样本、哪个头、query 位置 i，能否看 key 位置 j**"。
某维度长度为 1（或缺失），就表示该维度上**所有位置共享同一份掩码**——这正是各种简写形状的意义来源。

| mask 形状 | 哪些维度被共享 | 含义 | 典型场景 |
|---|---|---|---|
| `[B, H, L, L]` | 无共享，逐元素定义 | 每个样本每个头完全独立的掩码 | 极少用，最浪费内存；只有每个 head 真的需要不同可见性时才用 |
| `[B, 1, L, L]` | 所有头共享 | 每个样本一套掩码，H 个 head 用同一 mask | 自定义结构化 mask（如 BEV 中每个 query 只激活部分区域） |
| `[1, 1, L, L]` | batch 和头都共享 | 全 batch 共享的"位置关系"掩码 | 因果 mask（下三角）——位置关系与样本无关 |
| `[L, L]` | 同上（连维度都省） | 与 `[1,1,L,L]` 等价，最简洁写法 | 同上 |
| `[B, 1, 1, L]` | 头共享 + 所有 query 行共享 | **只按 key 屏蔽**：j 有效与否只取决于 key 位置，与谁在看它无关 | padding mask（屏蔽 padding 列），每行被广播复制成一样的 |
| `[B, 1, L, 1]` | 头共享 + 所有 key 列共享 | **只按 query 屏蔽**：整行要么全可见要么全屏蔽 | 少用；如某 query 位置本身无效（占位 token）不计损失时 |

**为什么这些形状广播后语义正确（图示）**

`[L, L]`（因果）从右对齐补两维 → `[1, 1, L, L]`，对 B、H 复制：
attn 的每个 `[L, L]` 块都套上同一个下三角 → "任何样本任何头，位置 i 都只能看 j<=i"。

`[B, 1, 1, L]`（padding）广播时复制到每一行 i：
```
      k0 k1 k2 k3 k4 k5        (该样本 k3..k5 是 padding)
 q0 [  1  1  1  0  0  0 ]
 q1 [  1  1  1  0  0  0 ]   ← 同一列模式复制到每个 query 行
 q2 [  1  1  1  0  0  0 ]
      → "所有 query 都看不到 padding 列"，只需 L 个数就能描述
```
而 `[B, 1, L, 1]` 则是复制到每一列：整行 i 同生共死，用来屏蔽 query 自身。

**一条经验法则**：掩码"取决于什么"，就保留什么维度；不取决于的维度就写 1（或省略）。

| 掩码只取决于 | 最小形状 |
|---|---|
| key 位置 j（padding） | `[B, 1, 1, L]` |
| query/key 位置对 (i, j)（因果） | `[L, L]` |
| 二者都要 | `[B, 1, L, L]`（见第 4 节叠加写法） |
| 还取决于头 | `[B, H, L, L]` |

**内存视角**：mask 是 bool 张量，`[B,H,L,L]` 在 B=8、H=16、L=4096 时要 8×16×4096² = 2G 元素（约 2 GB）；而 `[L,L]` 只要 16 MB，`[B,1,1,L]` 仅 KB 级。能不展开就不要展开——`masked_fill` 和 SDPA 都接受可广播的 mask。

### 3. `[L, L]` 方阵 mask 的语义与构造

**第 i 行第 j 列 = "query 位置 i 能否看 key 位置 j"**（行=query 维即倒数第二维，列=key 维即最后一维）。
自注意力 Q、K 来自同一序列，长度都是 L，所以方阵即可表达"谁看谁"。

**① 因果 mask（屏蔽未来）—— 下三角**

```python
i = torch.arange(L).view(-1, 1)   # [L, 1] 行下标
j = torch.arange(L).view(1, -1)   # [1, L] 列下标
mask = (i >= j)                   # 广播成 [L, L]，j <= i 才可见
# 等价写法: torch.tril(torch.ones(L, L))
```

```
      k0 k1 k2 k3 k4 k5
 q0 [  1  0  0  0  0  0 ]
 q1 [  1  1  0  0  0  0 ]   ← 每行只开放到自己这个位置
 q2 [  1  1  1  0  0  0 ]
```

**② padding mask —— 本质是 `[B, 1, 1, L]`，只屏蔽列**

```python
# valid_lens = [5, 3]，L = 6
mask = (torch.arange(L) < valid_lens.view(-1, 1))   # [B, L] → 补维 [B,1,1,L]
```

（d2l 版的 `valid_lens` + `masked_softmax` 表达的就是这个，只是折叠了 head 维。）

### 4. 实际 decoder：因果 ∧ padding 叠加 → 真·方阵

```python
causal = (i >= j)                        # [L, L]
pad    = valid.view(B, 1, 1, L).bool()   # [B, 1, 1, L]
mask   = causal.view(1, 1, L, L) & pad   # 广播相交 → [B, 1, L, L]
```

第 i 行的有效区 = "前 i 个位置中非 padding 的那些"。

### 5. 各场景该用哪种

| 场景 | mask |
|---|---|
| Encoder 自注意力（BERT） | 只用 padding mask `[B,1,1,L]`，无因果约束 |
| Decoder 自注意力（GPT） | 因果 `[L,L]`；有 padding 再叠加成 `[B,1,L,L]` |
| Cross-attention（Q 长 Lq，K 长 Lk） | 方阵变长方形 `[Lq, Lk]`，"行=query、列=key"规则不变 |

### 6. 坑点

- `masked_fill` 的 condition 必须是 **bool**：`masked_fill(mask==0, ...)` ✔，`masked_fill(mask, ...)`（0/1 float）✘ 报错。
- 若某行全被屏蔽（如 `valid_lens=0`），整行 `-inf` → softmax 出 **NaN**；工程上可改用 `-1e4`（d2l 即如此）。
- `F.scaled_dot_product_attention` 的 bool `attn_mask` 约定**相反**：True = 保留（float mask 则是直接加到 score 上，用 `-inf` 屏蔽）。从手写版迁移时注意取反。


## 官方 API 版本：`nn.MultiheadAttention` 参数详解

前面手写版对应的"生产写法"是 PyTorch 官方封装的 `nn.MultiheadAttention`，
内部集成了 QKV 融合投影（`in_proj_weight`）、多头变换、输出投影（`out_proj`），
且底层默认调用 `F.scaled_dot_product_attention` 的 fused kernel。

### 构造参数 `nn.MultiheadAttention(...)`

| 参数 | 含义 |
|---|---|
| `embed_dim` (E) | 输入 token 的特征维 d_model，必须能被 num_heads 整除 |
| `num_heads` (H) | 头数，每头 d_k = E/H |
| `dropout` | 作用在**注意力权重**上的 dropout（不是输出上），默认 0 |
| `bias` | 输入/输出投影是否带偏置，**默认 True**（手写版通常 bias=False） |
| `kdim` / `vdim` | **cross-attention** 专用：K、V 的输入特征维，默认 None=取 embed_dim。Q 和 K/V 来源不同、维度不同时就靠它 |
| `batch_first` | 默认 False，输入形状 `(L, N, E)`（seq 在前，RNN 传统）；True 才是 `(N, L, E)`。**忘设 batch_first 是最常见 bug**，不报错但语义全错 |
| `add_bias_kv` / `add_zero_attn` | 已废弃，不用管 |

### 前向参数 `forward(query, key, value, attn_mask=, key_padding_mask=, need_weights=, average_attn_weights=, is_causal=)`

| 参数 | 含义 |
|---|---|
| `query/key/value` | 三个独立输入（self-attn 时传同一个张量三次）——接口比手写版泛用，天然支持 cross-attn |
| `attn_mask` | 2D `(L_q, L_k)` 或 3D `(N*H, L_q, L_k)`。**float：直接加到 scores 上**（用 `-inf` 屏蔽）；**bool：True = 该位置不允许看** |
| `key_padding_mask` | `(N, L_k)`，bool：**True = 该 key 位置被忽略**。写 padding mask 的专用快捷通道，内部自动并入 attn_mask |
| `need_weights` | 默认 True 返回注意力权重；**为 True 时无法走 fused kernel**（要物化权重矩阵），训练时应设 False 提速 |
| `average_attn_weights` | True：返回对 H 头平均后的 `[N, L_q, L_k]`；False：返回逐头 `[N, H, L_q, L_k]`（做"可视化多头权重"练习用 False） |
| `is_causal` | 因果快捷开关。注意 **torch 2.0.x 要求同时传 attn_mask** 否则报错，≥2.1 可单独使用；稳妥做法是用 `generate_square_subsequent_mask(L)` 显式生成 |

### ⚠️ bool mask 约定的"三套语义"（本机实测，torch 2.0.1）

| API | bool True 的含义 |
|---|---|
| 手写 `masked_fill(mask, -inf)` | **屏蔽** |
| `nn.MultiheadAttention` 的 `attn_mask` / `key_padding_mask` | **屏蔽**（与 masked_fill 一致） |
| `F.scaled_dot_product_attention` 的 `attn_mask` | **保留**（与前两者相反！float 版则是加到 scores） |

从手写版迁移到 SDPA 时布尔 mask 必须**取反**，而迁移到 nn.MultiheadAttention 时不用——
这是最容易静默出错（结果"看起来正常"但注意力看反了）的地方。


In [ ]:
# ===== API 版本：直接调用 nn.MultiheadAttention（显式指定所有维度参数）=====
class MultiHeadAttentionAPI(nn.Module):
    """与手写版等价的包装：QKV投影/分头/输出投影全部由官方实现内部完成"""
    def __init__(self, embed_dim, num_heads, kdim=None, vdim=None, dropout=0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim,   # Q 的输入特征维，同时也是内部投影目标维
            num_heads=num_heads,   # 头数 H，每头 d_k = embed_dim/num_heads
            kdim=kdim,             # K 的输入特征维（cross-attn），None=embed_dim
            vdim=vdim,             # V 的输入特征维（cross-attn），None=embed_dim
            dropout=dropout,
            bias=False,            # 手写习惯；API 默认是 True
            batch_first=True,      # 输入 (N, L, E)；API 默认 False=(L, N, E)
        )

    def forward(self, queries, keys, values, attn_mask=None,
                key_padding_mask=None, need_weights=False):
        # self-attn 时 queries=keys=values；cross-attn 时来自两个不同序列
        out, weights = self.mha(queries, keys, values,
                                attn_mask=attn_mask,
                                key_padding_mask=key_padding_mask,
                                need_weights=need_weights,
                                average_attn_weights=False)  # False -> 逐头返回权重
        return out, weights   # out: (N, L_q, embed_dim); weights: None 若 need_weights=False

In [ ]:
X = torch.randn(2, 4, 100)

# --- 1. self-attn + padding mask：key_padding_mask (N, L_k)，bool 约定 True=屏蔽 ---
kpm = torch.tensor([[False, False, True, True],
                    [False, False, False, True]])   # 两个样本各自的 padding key
api1 = MultiHeadAttentionAPI(embed_dim=100, num_heads=5).eval()
out, w = api1(X, X, X, key_padding_mask=kpm, need_weights=True)
print('输出形状:', tuple(out.shape))      # (2, 4, 100)
print('逐头权重形状:', tuple(w.shape))    # (2, 5, 4, 4) = (N, H, L_q, L_k)
print('样本0的padding列权重为0:', torch.allclose(w[0, ..., 2:], torch.zeros_like(w[0, ..., 2:])))
print('样本1仅末列为0:', torch.allclose(w[1, ..., 3:], torch.zeros_like(w[1, ..., 3:])))

# --- 2. 因果 mask：float 加法版（0 可见 / -inf 屏蔽） ---
L = 4
i, j = torch.arange(L).view(-1, 1), torch.arange(L).view(1, -1)
causal_float = torch.where(i < j, torch.tensor(float('-inf')), torch.tensor(0.0))
api2 = MultiHeadAttentionAPI(embed_dim=100, num_heads=5).eval()
out_c, w_c = api2(X, X, X, attn_mask=causal_float, need_weights=True)
print('上三角权重为0:', torch.allclose(w_c.triu(diagonal=1), torch.zeros_like(w_c)))

# --- 3. cross-attn：Q 与 K/V 输入维度不同，显式用 kdim/vdim 指定 ---
q_cross = torch.randn(2, 4, 100)      # (N, L_q, embed_dim=100)
memory  = torch.randn(2, 7, 64)       # (N, L_k, kdim=64) K/V 来自 encoder，维度 64
api3 = MultiHeadAttentionAPI(embed_dim=100, num_heads=5, kdim=64, vdim=64).eval()
out_x, w_x = api3(q_cross, memory, memory, need_weights=True)   # 要看权重必须显式开
print('cross-attn 输出形状:', tuple(out_x.shape))   # (2, 4, 100) 长度跟 Q，维度跟 embed_dim
print('cross-attn 权重形状:', tuple(w_x.shape))     # (2, 5, 4, 7) L_q x L_k
# 此时内部权重分三块: q_proj [100,100] / k_proj [100,64] / v_proj [100,64]
print('k_proj_weight 形状:', tuple(api3.mha.k_proj_weight.shape))

# --- 4. 数值一致性：与 d2l 手写版权重移植后逐元素对齐 ---
ref = MultiHeadAttention(100, 100, 100, 100, 5, dropout=0.0).eval()   # 前面的手写版
api = MultiHeadAttentionAPI(embed_dim=100, num_heads=5).eval()
with torch.no_grad():
    # embed_dim==kdim==vdim 时 QKV 融合成一整块 in_proj_weight [3E, E]，顺序 q|k|v
    api.mha.in_proj_weight.copy_(torch.cat([ref.W_q.weight, ref.W_k.weight, ref.W_v.weight]))
    api.mha.out_proj.weight.copy_(ref.W_o.weight)

X6 = torch.randn(2, 6, 100)
o_ref = ref(X6, X6, X6, None)        # 手写版（self-attn，无mask）
o_api, _ = api(X6, X6, X6)
print('手写版与API版输出一致:', torch.allclose(o_ref, o_api, atol=1e-5))